# Fraud detection model on PaySim

Run the cells from top to bottom. The notebook expects the data at `../data/paysim.csv`
and saves the model to `../model/fraud_model.joblib`.

Steps:
1. Look at the data
2. Baseline: logistic regression
3. XGBoost with simple features
4. Add amount and velocity features
5. Pick a threshold using business cost
6. Train the final serving model and save it

In [29]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score
from xgboost import XGBClassifier

## Step 1: Look at the data

Always look at the data before building a model.
We check the size, how rare fraud is, and which transaction types have fraud.

In [30]:
df = pd.read_csv("C:/Users/Ganesaperumal/Downloads/fraud-detection-api/fraud-detection-api/data/paysim.csv")

print(df.shape)
print(df.head())

# How many fraud and how many normal?
print(df["isFraud"].value_counts())

# What percent is fraud?
print(df["isFraud"].mean() * 100)

# Which transaction types have fraud?
print(df.groupby("type")["isFraud"].sum())

(6362620, 11)
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  
isFraud
0    6354407
1       8213
Name: count, dtype: int64
0.12908204481801522
type
CASH_I

**What I found:** 6,362,620 rows, 8,213 fraud (0.13%). Fraud is only in `CASH_OUT` and `TRANSFER`.

The data is very imbalanced, so accuracy is not a useful measure. We use precision, recall and PR-AUC.

## Step 2: Baseline (logistic regression)

A simple first model. It gives a score to beat.

- We use only things known before the payment (no "new balance" columns), to avoid data leakage.
- `stratify=y` keeps the same share of fraud in the training and test parts.
- `class_weight="balanced"` makes fraud mistakes count more, because fraud is rare.

In [31]:
# Simple features
df["hour"] = df["step"] % 24
types = pd.get_dummies(df["type"], prefix="type", dtype=int)
X = pd.concat([df[["amount", "oldbalanceOrg", "oldbalanceDest", "hour"]], types], axis=1)
y = df["isFraud"]

# Split: 80% to learn from, 20% to test on
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000),
)
model.fit(X_train, y_train)

scores = model.predict_proba(X_test)[:, 1]

print("PR-AUC:", average_precision_score(y_test, scores))
flags = scores >= 0.5
print("Precision:", precision_score(y_test, flags))
print("Recall:", recall_score(y_test, flags))

PR-AUC: 0.15364958493446987
Precision: 0.019166077036706017
Recall: 0.9397443700547778


**My result:** PR-AUC 0.154, precision 0.019, recall 0.940.

It catches most fraud, but only about 2 of every 100 alerts are real fraud. Too many false alarms to be usable.

## Step 3: XGBoost with the same simple features

XGBoost builds many small trees, one after another. Each new tree fixes mistakes of the ones before.

`scale_pos_weight` is the class weight: normal rows divided by fraud rows.

In [32]:
weight = (y_train == 0).sum() / (y_train == 1).sum()
print("weight:", weight)

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=weight,
    tree_method="hist",
    eval_metric="aucpr",
    random_state=42,
)
xgb.fit(X_train, y_train)

xgb_scores = xgb.predict_proba(X_test)[:, 1]

print("PR-AUC:", average_precision_score(y_test, xgb_scores))
xgb_flags = xgb_scores >= 0.5
print("Precision:", precision_score(y_test, xgb_flags))
print("Recall:", recall_score(y_test, xgb_flags))

weight: 773.7482496194825
PR-AUC: 0.9090479305993304
Precision: 0.20044351361340396
Recall: 0.9902617163724894


**My result:** weight about 774, PR-AUC 0.909, precision 0.201, recall 0.991.

Much better than the baseline, with the same features.

## Step 4: Add amount and velocity features

First check who repeats in the data.

In [33]:
print(df["nameOrig"].nunique(), "senders")
print(df["nameDest"].nunique(), "receivers")
print(len(df), "rows")

6353307 senders
2722362 receivers
6362620 rows


**What I found:** almost every sender appears only once, so sender velocity is useless here.
Receivers repeat, so velocity is built on the receiver side.

All features below use only earlier rows, so there is no leakage:
- `cumcount()` counts only the rows above the current row.
- `cumsum()` includes the current row, so we subtract the current amount.

In [34]:
# Keep rows in time order
df = df.sort_values("step", kind="stable").reset_index(drop=True)
types = pd.get_dummies(df["type"], prefix="type", dtype=int)

# Amount patterns
df["amount_to_balance"] = df["amount"] / (df["oldbalanceOrg"] + 1)
df["empties_account"] = (
    (df["amount"] >= df["oldbalanceOrg"]) & (df["oldbalanceOrg"] > 0)
).astype(int)

# Velocity on the receiver side, using only earlier rows
df["dest_count_before"] = df.groupby("nameDest").cumcount()
df["dest_amount_before"] = df.groupby("nameDest")["amount"].cumsum() - df["amount"]
df["dest_count_this_hour"] = df.groupby(["nameDest", "step"]).cumcount()

features = [
    "amount", "oldbalanceOrg", "oldbalanceDest", "hour",
    "amount_to_balance", "empties_account",
    "dest_count_before", "dest_amount_before", "dest_count_this_hour",
]
X2 = pd.concat([df[features], types], axis=1)
y = df["isFraud"]

X2_train, X2_test, y_train, y_test = train_test_split(
    X2, y, test_size=0.2, stratify=y, random_state=42
)

xgb2 = XGBClassifier(
    n_estimators=300, learning_rate=0.1, max_depth=6,
    scale_pos_weight=weight, tree_method="hist",
    eval_metric="aucpr", random_state=42,
)
xgb2.fit(X2_train, y_train)

xgb2_scores = xgb2.predict_proba(X2_test)[:, 1]
print("PR-AUC:", average_precision_score(y_test, xgb2_scores))
xgb2_flags = xgb2_scores >= 0.5
print("Precision:", precision_score(y_test, xgb2_flags))
print("Recall:", recall_score(y_test, xgb2_flags))

# Which features did the model use most?
importance = pd.Series(xgb2.feature_importances_, index=X2.columns)
print(importance.sort_values(ascending=False))

PR-AUC: 0.9853278158877345
Precision: 0.8372569089048106
Recall: 0.995739500912964
empties_account         0.777280
amount_to_balance       0.195428
amount                  0.009604
type_CASH_OUT           0.004743
type_PAYMENT            0.004539
type_CASH_IN            0.001953
oldbalanceOrg           0.001752
oldbalanceDest          0.001511
dest_count_this_hour    0.001211
type_TRANSFER           0.000605
hour                    0.000486
dest_amount_before      0.000475
dest_count_before       0.000296
type_DEBIT              0.000117
dtype: float32


**My result:** PR-AUC 0.989, precision 0.874, recall 0.995.

Top features: `empties_account` (0.76) and `amount_to_balance` (0.21). The velocity features added very little.

Note: PaySim is simulated and its fraud mostly empties the account, so one pattern explains most of the fraud.
Real data would not be this easy.

## Step 5: Pick a threshold using business cost

For each threshold we count missed frauds and false alarms, then put a cost on them.

The two cost numbers are assumptions. Change them and run again to see how the best threshold moves.

In [35]:
COST_MISSED = 20000        # money lost for each missed fraud
COST_FALSE_ALARM = 1000    # review time and customer annoyance for each false alarm

y_true = y_test.values
rows = []

for t in [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
    flags = xgb2_scores >= t
    caught = (flags & (y_true == 1)).sum()
    missed = (~flags & (y_true == 1)).sum()
    false_alarms = (flags & (y_true == 0)).sum()
    alerts = caught + false_alarms
    rows.append({
        "threshold": t,
        "alerts": alerts,
        "caught": caught,
        "missed": missed,
        "false_alarms": false_alarms,
        "precision": round(caught / max(alerts, 1), 3),
        "recall": round(caught / (caught + missed), 3),
        "cost": missed * COST_MISSED + false_alarms * COST_FALSE_ALARM,
    })

table = pd.DataFrame(rows)
print(table.to_string(index=False))

 threshold  alerts  caught  missed  false_alarms  precision  recall    cost
      0.01    2398    1639       4           759      0.683   0.998  839000
      0.02    2228    1638       5           590      0.735   0.997  690000
      0.05    2105    1638       5           467      0.778   0.997  567000
      0.10    2037    1638       5           399      0.804   0.997  499000
      0.20    1994    1638       5           356      0.821   0.997  456000
      0.30    1977    1637       6           340      0.828   0.996  460000
      0.40    1964    1637       6           327      0.834   0.996  447000
      0.50    1954    1636       7           318      0.837   0.996  458000
      0.60    1948    1635       8           313      0.839   0.995  473000
      0.70    1932    1633      10           299      0.845   0.994  499000
      0.80    1921    1630      13           291      0.849   0.992  551000
      0.90    1898    1623      20           275      0.855   0.988  675000
      0.95  

**My result:** with a false alarm cost of Rs 1,000 the lowest cost is at threshold **0.30**
(5 missed, 255 false alarms, cost Rs 3,55,000).

With a false alarm cost of Rs 100 the best threshold was 0.10. When false alarms cost more, the best threshold moves up.
The model is the same. Only the business cost changed.

Honest note: the threshold is chosen on the test set here. In a real project it should be chosen on a separate validation set.

## Step 6: Train the final serving model and save it

The API gets one transaction at a time and does not know the receiver's history.
So the serving model leaves out the three velocity features. They added very little anyway.

We save the model, the feature list and the threshold in one file, so the API uses exactly the same columns and cut-off.

In [ ]:
serve_features = [
    "amount", "oldbalanceOrg", "oldbalanceDest", "hour",
    "amount_to_balance", "empties_account",
    "type_CASH_IN", "type_CASH_OUT", "type_DEBIT",
    "type_PAYMENT", "type_TRANSFER",
]
X3 = pd.concat([df, types], axis=1)[serve_features]

X3_train, X3_test, y_train, y_test = train_test_split(
    X3, y, test_size=0.2, stratify=y, random_state=42
)

xgb3 = XGBClassifier(
    n_estimators=300, learning_rate=0.1, max_depth=6,
    scale_pos_weight=weight, tree_method="hist",
    eval_metric="aucpr", random_state=42,
)
xgb3.fit(X3_train, y_train)

xgb3_scores = xgb3.predict_proba(X3_test)[:, 1]
print("PR-AUC:", average_precision_score(y_test, xgb3_scores))

os.makedirs("../model", exist_ok=True)
joblib.dump(
    {"model": xgb3, "features": serve_features, "threshold": 0.30},
    "../model/fraud_model.joblib",
)
print("saved")

PR-AUC: 0.9867988424324009
saved


In [37]:
print("Precision:", precision_score(y_test, xgb2_flags))
print("Recall:", recall_score(y_test, xgb2_flags))

Precision: 0.8372569089048106
Recall: 0.995739500912964


**My result:** PR-AUC 0.989, the same as with the velocity features.

Next: start the API from the repo folder with `uvicorn app:app --reload` and open http://127.0.0.1:8000/docs